# Download Common Voice 26.0 (Swahili) from Mozilla Data Collective

Uses the official `datacollective` Python library to authenticate with and
download dataset `cmqim4c1000tmnr07zq3vwhor`
(slug `common-voice-scripted-speech-26-0-swahil-0228b2f6`) from the Mozilla
Data Collective (MDC) API.

**Usage instructions:**
1. Install the `datacollective` package.
2. Set your API key as an environment variable (`MDC_API_KEY`).
3. Run `download_dataset(...)` and/or `load_dataset(...)`.

The Dataset ID and Dataset Slug are interchangeable everywhere below -- use
whichever you find easier to keep track of.

**Before running, in this notebook's settings:** Settings -> Internet -> On
(required; MDC is an external API).

## 1. Install the `datacollective` package

In [ ]:
!pip install -q datacollective

## 2. Set your API key as an environment variable

Generate a key at https://mozilladatacollective.com after creating an account
and agreeing to this dataset's Terms & Conditions on its MDC page.

**Recommended on Kaggle:** store it as a Kaggle Secret (Add-ons -> Secrets)
named `MDC_API_KEY` and load it at runtime as below, rather than typing the
raw key into a cell -- notebook cells and their outputs can end up saved,
shared, or version-controlled.
*If this key was ever pasted anywhere outside a secrets vault (chat, a
script, a committed file), rotate/revoke it on the MDC platform first.*

Outside Kaggle, the library also reads `MDC_API_KEY` from a `.env` file in
the working directory (via `python-dotenv`) or from the shell environment --
either works with the exact same `download_dataset`/`load_dataset` calls
below, no code changes needed.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["MDC_API_KEY"] = user_secrets.get_secret("MDC_API_KEY")
print("MDC_API_KEY loaded from Kaggle Secrets.")

## 3. Run the download / load commands

`download_dataset` fetches the raw archive to local disk and returns its path
(resumes automatically if re-run after an interruption).

In [ ]:
from datacollective import download_dataset

DATASET_ID = "cmqim4c1000tmnr07zq3vwhor"  # or the slug: "common-voice-scripted-speech-26-0-swahil-0228b2f6"
# Explicit directory: the SDK's own default (~/.mozdata/datasets, i.e.
# /root/.mozdata/datasets on Kaggle) lives outside every Kaggle-persisted
# area and is wiped when the session ends. /kaggle/working is at least
# eligible for "Save Version"; see the disk-space note below for why even
# that isn't enough on its own to avoid re-downloading in a future session.
DOWNLOAD_DIR = "/kaggle/working/mdc_common_voice_sw"

dataset_path = download_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR)
print("Downloaded archive to:", dataset_path)

`load_dataset` downloads (if needed), extracts, and parses the dataset via
MDC's schema registry, returning a pandas DataFrame directly -- if this
dataset has a registered schema. If not, it raises `RuntimeError` and you
fall back to the archive already fetched above.

In [ ]:
from datacollective import load_dataset

# Load as a Pandas DataFrame, if supported!
try:
    dataset = load_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR)
    print(dataset.shape)
    display(dataset.head())
except RuntimeError as e:
    print("No registered schema for this dataset yet -- extract dataset_path manually instead:")
    print(e)

## Fallback: manual extraction

Only needed if `load_dataset` raised `RuntimeError` above. Extracts the
archive downloaded by `download_dataset` and locates the transcript TSV that
`scripts/select_subset.py` in the Swahili-Deepfake-dataset pipeline consumes
(columns `path`, `client_id`, `sentence` by default).

In [ ]:
import tarfile
from pathlib import Path
import pandas as pd

extract_dir = Path(dataset_path).parent / "extracted"
extract_dir.mkdir(exist_ok=True)

with tarfile.open(dataset_path) as tar:
    tar.extractall(path=extract_dir)

tsv_candidates = list(extract_dir.rglob("validated.tsv")) or list(extract_dir.rglob("*.tsv"))
print("Found TSV files:")
for p in tsv_candidates:
    print(" -", p)

if tsv_candidates:
    preview = pd.read_csv(tsv_candidates[0], sep="\t", nrows=5)
    display(preview)
    print("Columns:", list(preview.columns))

## Disk space and persisting across sessions

The archive is roughly 20-22 GB compressed; extraction needs comparable free
space again on top of that.

**Nothing outside `/kaggle/working` (via Save Version) or an attached
`/kaggle/input` Dataset survives when a session ends** -- including the
SDK's own default cache directory (`~/.mozdata/datasets`) and `/kaggle/temp`.
`DOWNLOAD_DIR` above already points at `/kaggle/working` so the file is at
least eligible to persist, but that alone still isn't enough: `/kaggle/working`
only persists if you click **Save Version**, and it has a smaller output quota
than this dataset (compressed + extracted) may fit in.

**To avoid re-downloading in every future session:**

1. Run this notebook once and let the download/extraction finish.
2. Click **Save Version** to commit the notebook and its `/kaggle/working` output.
3. From the notebook's output/Data pane, use **"New Dataset"** to publish
   that output as a private Kaggle Dataset (uploads from Kaggle's own
   storage, not back through your connection).
4. In any future notebook, **Add Input -> your new dataset** -- it mounts
   read-only at `/kaggle/input/<dataset-slug>/` instantly, no MDC download.

If you're tight on space and don't need to keep the compressed archive after
extracting, delete it:
```python
# import os
# os.remove(dataset_path)
```